In [1]:
import pandas as pd

In [2]:
df_train = pd.read_csv("preprocessed/train.csv")
df_test = pd.read_csv("preprocessed/test.csv")

In [3]:
len(df_train)

500000

In [4]:
len(df_test)

85000

In [5]:
df_train.head()

,user_id,component0,component1,component2,component3,component4,component5,component6,component7,component8,...,country_id,region_id,timezone_id,domain,path,n_browsers,n_os,n_domains,n_paths,target
0,00006a10d9ff5f9d312029ac4d296430,10214.0,-1254.0,10495.0,7705.0,9547.0,5801.0,864.0,8837.0,5312.0,...,c31b4e,470e75,f6155e,6a81948,NaN,1,1,1,0,1
1,00008d5fd6bff19884138510b077a22b,10214.0,-1254.0,10495.0,7705.0,9547.0,5801.0,864.0,8837.0,5312.0,...,c31b4e,1fbfa5,e56e80,6a81948,NaN,1,1,1,0,1
2,000098f8aa8b68a125148caff0a02827,15633.0,22503.0,-2155.0,-557.0,2232.0,16084.0,-87.0,-2820.0,6578.0,...,e48e27,NaN,f0c3df,b53adc6,NaN,1,1,1,0,0
3,0000b0ec7cb23e402cc5bb0487deef4c,12055.5,10011.5,10496.0,-1075.0,9866.5,8561.5,-249.0,-7344.5,12256.5,...,c31b4e,97023a,e56e80,b56ea20,/12db98f2,2,2,2,1,1
4,0000be6a07f2fcfd68b8486d309e3a22,-3578.0,2164.0,-12993.0,5330.0,5018.0,8603.0,-2646.0,8588.0,-30424.0,...,c31b4e,380ef0,e56e80,9f1218f,NaN,1,1,1,0,1


In [6]:
df_test.head()

,user_id,component0,component1,component2,component3,component4,component5,component6,component7,component8,...,os_version,country_id,region_id,timezone_id,domain,path,n_browsers,n_os,n_domains,n_paths
0,000098f8aa8b68a125148caff0a02827,15633.0,22503.0,-2155.0,-557.0,2232.0,16084.0,-87.0,-2820.0,6578.0,...,10,e48e27,NaN,f0c3df,b53adc6,NaN,1,1,1,0
1,0000eae7d06e8c6033731d1c1ba0c382,1489.0,15548.0,1283.0,5662.0,5217.0,11942.0,18743.0,12438.0,-2706.0,...,11,c31b4e,37246e,b4a275,72879b4,/15fd854f,1,1,1,1
2,0000f7fa001a8d49b8e83b91a1215e17,18530.0,14117.0,9243.0,-3607.0,-1048.0,5971.0,-5180.0,-5011.0,13938.0,...,10,9bd92f,NaN,100cd5d,b6630fd,NaN,1,1,1,0
3,0001c39409954fe4b4148a23154fa905,15290.0,-740.0,12145.0,11199.0,6168.0,5630.0,13574.0,10086.0,917.0,...,10,c31b4e,380ef0,e56e80,635e50c,/176eef88,1,1,1,1
4,00029eeaef3671876a5d119acff7e792,19051.0,9021.0,9045.0,-713.0,7571.0,10582.0,4131.0,14754.0,-2675.0,...,10,c31b4e,f66ff,f6155e,7f91a1d,/1283abdf,1,1,1,1


In [7]:
categorical_columns = [
    "browser",
    "os",
    "country_id",
    "region_id",
    "timezone_id",
    "domain",
    "event_count",
]

component_columns = [f"component{i}" for i in range(10)] + [
    "n_browsers",
    "n_os",
    "n_domains",
    "n_paths",
]

df_train[categorical_columns] = df_train[categorical_columns].fillna("NaN").astype(str)
df_test[categorical_columns] = df_test[categorical_columns].fillna("NaN").astype(str)

df_train["target"] = df_train["target"].astype("float32")

In [8]:
categorical_maps = {}
categorical_dims = {}

for col in categorical_columns:
    unique_vals = list(df_train[col].unique())
    mp = {v: i for i, v in enumerate(unique_vals)}
    if "NaN" not in mp:
        mp["NaN"] = len(mp)
    categorical_maps[col] = mp
    categorical_dims[col] = len(mp)

In [9]:
def encode_categorical(df, columns, maps):
    import numpy as np

    arr = np.zeros((len(df), len(columns)), dtype=np.int64)
    for i, col in enumerate(columns):
        mp = maps[col]
        arr[:, i] = df[col].apply(lambda x, mp=mp: mp[x] if x in mp else mp["NaN"])
    return arr

In [10]:
df_all_train_x = df_train[categorical_columns + component_columns]
df_all_train_user_id = df_train["user_id"].copy()
df_all_train_y = df_train["target"].copy()

df_all_train_cat_x = encode_categorical(
    df_all_train_x, categorical_columns, categorical_maps
)
df_all_train_num_x = df_all_train_x[component_columns].astype("float32").values

In [11]:
df_test_x = df_test[categorical_columns + component_columns]
df_test_user_id = df_test["user_id"].copy()

df_test_cat_x = encode_categorical(df_test_x, categorical_columns, categorical_maps)
df_test_num_x = df_test_x[component_columns].astype("float32").values

In [12]:
from sklearn.model_selection import train_test_split

df_train_cat_x, df_val_cat_x, df_train_num_x, df_val_num_x, df_train_y, df_val_y = (
    train_test_split(
        df_all_train_cat_x,
        df_all_train_num_x,
        df_all_train_y.values,
        test_size=0.2,
        random_state=42,
        stratify=df_all_train_y.values,
    )
)

In [13]:
categorical_maps["os"]

{'Android': 0,
 'iOS': 1,
 'Windows': 2,
 'Mac OS X': 3,
 'Tizen': 4,
 'Linux': 5,
 'Chrome OS': 6,
 'Ubuntu': 7,
 'Windows Phone': 8,
 'Fedora': 9,
 'FreeBSD': 10,
 'Other': 11,
 'Chromecast': 12,
 'NaN': 13}

In [14]:
categorical_dims["os"]

14

In [15]:
for i, column in enumerate(categorical_columns):
    print(
        "max_id_in_train=",
        df_train_cat_x[:, i].max(),
        "max_id_in_val=",
        df_val_cat_x[:, i].max(),
        column,
        categorical_dims[column],
    )

max_id_in_train= 59 max_id_in_val= 52 browser 60
max_id_in_train= 13 max_id_in_val= 11 os 14
max_id_in_train= 150 max_id_in_val= 149 country_id 152
max_id_in_train= 258 max_id_in_val= 258 region_id 259
max_id_in_train= 217 max_id_in_val= 216 timezone_id 219
max_id_in_train= 4845 max_id_in_val= 4837 domain 4847
max_id_in_train= 8 max_id_in_val= 8 event_count 10


In [16]:
df_train_cat_x = df_train_cat_x.astype("int64")
df_train_num_x = df_train_num_x.astype("float32")
df_train_y = df_train_y.astype("float32")

In [17]:
df_all_train_cat_x.shape, df_all_train_num_x.shape, df_all_train_y.shape

((500000, 7), (500000, 14), (500000,))

In [18]:
df_train_cat_x.shape, df_train_num_x.shape, df_train_y.shape

((400000, 7), (400000, 14), (400000,))

In [19]:
df_val_cat_x.shape, df_val_num_x.shape, df_val_y.shape

((100000, 7), (100000, 14), (100000,))

In [20]:
import torch

df_all_train_cat_x_t = torch.tensor(df_all_train_cat_x, dtype=torch.int64)
df_all_train_num_x_t = torch.tensor(df_all_train_num_x, dtype=torch.float32)
df_train_cat_x_t = torch.tensor(df_train_cat_x, dtype=torch.int64)
df_train_num_x_t = torch.tensor(df_train_num_x, dtype=torch.float32)
df_train_y_t = torch.tensor(df_train_y, dtype=torch.float32)

df_test_cat_x_t = torch.tensor(df_test_cat_x, dtype=torch.int64)
df_test_num_x_t = torch.tensor(df_test_num_x, dtype=torch.float32)

In [21]:
from torch.utils.data import Dataset


class CustomDataset(Dataset):
    def __init__(self, categorical_features, numeric_features, targets):
        self.categorical_features = categorical_features
        self.numeric_features = numeric_features
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        return (
            self.categorical_features[index],
            self.numeric_features[index],
            self.targets[index],
        )

In [22]:
from torch.utils.data import DataLoader

BATCH_SIZE = 1024

ds_train = CustomDataset(df_train_cat_x, df_train_num_x, df_train_y)
ds_val = CustomDataset(df_val_cat_x, df_val_num_x, df_val_y)

train_loader = DataLoader(
    ds_train, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True
)
val_loader = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

In [23]:
import math


def make_embedding_dim(n_cat):
    return min(50, int(math.sqrt(n_cat)))


categorical_dims_list = [categorical_dims[column] for column in categorical_columns]
embedding_dims_list = [make_embedding_dim(n) for n in categorical_dims_list]
count_numeric_features = df_train_num_x.shape[1]

In [24]:
categorical_dims_list

[60, 14, 152, 259, 219, 4847, 10]

In [25]:
embedding_dims_list

[7, 3, 12, 16, 14, 50, 3]

In [26]:
count_numeric_features

14

In [27]:
import torch
from torch import nn


class CustomNet(nn.Module):
    def __init__(self, categorical_dims, embedding_dims, num_numeric_features):
        super().__init__()

        self.cat_embedding_layers = nn.ModuleList(
            [
                nn.Embedding(categorical_dims[i], embedding_dims[i])
                for i in range(len(categorical_dims))
            ]
        )

        self.embedding_dropout = nn.Dropout(0.05)

        self.numeric_batch_norm = nn.BatchNorm1d(num_numeric_features)

        self.combined_batch_norm = nn.BatchNorm1d(
            sum(embedding_dims) + num_numeric_features
        )

        self.embedding_total_dim = sum(embedding_dims) + num_numeric_features

        self.backbone = nn.Sequential(
            nn.Linear(self.embedding_total_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
        )

        self.head = nn.Sequential(
            nn.Linear(32, 256),
            nn.LeakyReLU(negative_slope=0.01),
            nn.Dropout(0.05),
            nn.Linear(256, 128),
            nn.LeakyReLU(negative_slope=0.01),
            nn.Dropout(0.05),
            nn.Linear(128, 64),
            nn.LeakyReLU(negative_slope=0.01),
            nn.Dropout(0.05),
            nn.Linear(64, 1),
        )

    def forward(self, categorical_x, numeric_x):
        emb_list = [
            emb(categorical_x[:, i]) for i, emb in enumerate(self.cat_embedding_layers)
        ]
        emb_cat = torch.cat(emb_list, dim=1)
        emb_cat = self.embedding_dropout(emb_cat)
        numeric_x = self.numeric_batch_norm(numeric_x.float())
        x = torch.cat([emb_cat, numeric_x], dim=1)
        x = self.combined_batch_norm(x)
        bb = self.backbone(x)
        return self.head(bb).squeeze(1)

    def extract_backbone(self, categorical_x, numeric_x):
        emb_list = [
            emb(categorical_x[:, i]) for i, emb in enumerate(self.cat_embedding_layers)
        ]
        emb_cat = torch.cat(emb_list, dim=1)
        emb_cat = self.embedding_dropout(emb_cat)

        numeric_x = self.numeric_batch_norm(numeric_x.float())

        x = torch.cat([emb_cat, numeric_x], dim=1)
        x = self.combined_batch_norm(x)

        return self.backbone(x)

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 20

print(categorical_dims_list, embedding_dims_list, count_numeric_features)
model = CustomNet(
    categorical_dims_list, embedding_dims_list, count_numeric_features
).to(device)
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

[60, 14, 152, 259, 219, 4847, 10] [7, 3, 12, 16, 14, 50, 3] 14


In [29]:
model

CustomNet(
  (cat_embedding_layers): ModuleList(
    (0): Embedding(60, 7)
    (1): Embedding(14, 3)
    (2): Embedding(152, 12)
    (3): Embedding(259, 16)
    (4): Embedding(219, 14)
    (5): Embedding(4847, 50)
    (6): Embedding(10, 3)
  )
  (embedding_dropout): Dropout(p=0.05, inplace=False)
  (numeric_batch_norm): BatchNorm1d(14, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (combined_batch_norm): BatchNorm1d(119, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (backbone): Sequential(
    (0): Linear(in_features=119, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
  )
  (head): Sequential(
    (0): Linear(in_features=32, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Dropout(p=0.05, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4

In [30]:
def find_best_threshold(model, loader):
    probs, targets = evaluate(model, loader, return_raw=True)

    best_thr = 0.5
    best_acc = 0.0

    for thr in np.linspace(0.05, 0.95, 200):
        predictions = (probs >= thr).astype(int)
        acc = accuracy_score(targets, predictions)
        if acc > best_acc:
            best_acc = acc
            best_thr = thr

    return best_thr, best_acc

In [31]:
import numpy as np
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score


def evaluate(model, loader, return_raw=False):
    model.eval()
    all_logits = []
    all_targets = []

    with torch.no_grad():
        for c, n, y in loader:
            c = c.to(device)
            n = n.to(device)
            y = y.to(device)

            logits = model(c, n)
            all_logits.append(logits.cpu())
            all_targets.append(y.cpu())

    logits = torch.cat(all_logits).numpy()
    targets = torch.cat(all_targets).numpy()

    probabilities = 1 / (1 + np.exp(-logits))

    if return_raw:
        return probabilities, targets

    return (
        log_loss(targets, probabilities),
        roc_auc_score(targets, probabilities),
        find_best_threshold(model, loader)[1],
    )

In [32]:
# model.load_state_dict(torch.load("models/best_model.pt"))

In [33]:
from tqdm.auto import tqdm

patience = 1
best_auc = -1
wait = 0

for epoch in range(EPOCHS):
    model.train()
    losses = []

    for c, n, y in tqdm(train_loader, desc=f"Epoch {epoch + 1}"):
        c = c.to(device)
        n = n.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(c, n)
        loss = loss_function(logits, y)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    train_loss = np.mean(losses)
    val_loss, val_auc, val_acc = evaluate(model, val_loader)

    print(
        f"epoch={epoch + 1} train={train_loss:.4f} val_loss={val_loss:.4f} auc={val_auc:.4f} acc={val_acc:.4f}"
    )

    if val_auc > best_auc:
        best_auc = val_auc
        wait = 0
        torch.save(model.state_dict(), "models/best_model.pt")
    else:
        wait += 1
        print(f"Wait {wait}/{patience}")
        if wait >= patience:
            print("Early stopping")
            break

Epoch 1:   0%|          | 0/391 [00:00<?, ?it/s]

epoch=1 train=0.5625 val_loss=0.4820 auc=0.8493 acc=0.7804


Epoch 2:   0%|          | 0/391 [00:00<?, ?it/s]

epoch=2 train=0.4685 val_loss=0.4452 auc=0.8739 acc=0.8041


Epoch 3:   0%|          | 0/391 [00:00<?, ?it/s]

...

epoch=18 train=0.3805 val_loss=0.3772 auc=0.9095 acc=0.8431


Epoch 19:   0%|          | 0/391 [00:00<?, ?it/s]

epoch=19 train=0.3784 val_loss=0.3770 auc=0.9098 acc=0.8440


Epoch 20:   0%|          | 0/391 [00:00<?, ?it/s]

epoch=20 train=0.3771 val_loss=0.3744 auc=0.9105 acc=0.8443


In [34]:
model.load_state_dict(torch.load("models/best_model.pt"))

<All keys matched successfully>

In [35]:
best_thr, best_acc = find_best_threshold(model, val_loader)
print(f"best_thr={best_thr:.4f} best_acc={best_acc:.4f}")
evaluate(model, val_loader)

best_thr=0.5656 best_acc=0.8443


(0.3743768617657129, 0.9104921562520492, 0.84431)

In [36]:
import pandas as pd
import torch

model.eval()
with torch.inference_mode():
    bb_train = (
        model.extract_backbone(
            df_all_train_cat_x_t.to(device), df_all_train_num_x_t.to(device)
        )
        .cpu()
        .numpy()
    )

    bb_test = (
        model.extract_backbone(df_test_cat_x_t.to(device), df_test_num_x_t.to(device))
        .cpu()
        .numpy()
    )

train_features = pd.concat(
    [
        df_all_train_user_id.reset_index(drop=True),
        df_all_train_x.reset_index(drop=True),
        pd.DataFrame(bb_train, columns=[f"bb_{i}" for i in range(bb_train.shape[1])]),
        df_all_train_y.reset_index(drop=True),
    ],
    axis=1,
)

test_features = pd.concat(
    [
        df_test_user_id.reset_index(drop=True),
        df_test_x.reset_index(drop=True),
        pd.DataFrame(bb_test, columns=[f"bb_{i}" for i in range(bb_test.shape[1])]),
    ],
    axis=1,
)

In [37]:
train_features

,user_id,browser,os,country_id,region_id,timezone_id,domain,event_count,component0,component1,...,bb_23,bb_24,bb_25,bb_26,bb_27,bb_28,bb_29,bb_30,bb_31,target
0,00006a10d9ff5f9d312029ac4d296430,YandexSearch,Android,c31b4e,470e75,f6155e,6a81948,1,10214.0,-1254.0,...,0.141555,0.519237,0.126132,0.0,0.034210,0.024856,0.135293,0.492822,0.171879,1.0
1,00008d5fd6bff19884138510b077a22b,Yandex Browser,iOS,c31b4e,1fbfa5,e56e80,6a81948,1,10214.0,-1254.0,...,0.116888,0.440709,0.015529,0.0,0.000000,0.048866,0.190242,0.542757,0.099703,1.0
2,000098f8aa8b68a125148caff0a02827,Chrome,Windows,e48e27,NaN,f0c3df,b53adc6,1,15633.0,22503.0,...,0.223871,0.366513,0.000000,0.0,0.000000,0.000000,0.000000,0.377329,0.000000,0.0
3,0000b0ec7cb23e402cc5bb0487deef4c,Chrome,Android,c31b4e,97023a,e56e80,b56ea20,2,12055.5,10011.5,...,0.376081,0.102755,0.217579,0.0,0.126420,0.000000,0.000000,0.298370,0.000000,1.0
4,0000be6a07f2fcfd68b8486d309e3a22,Chrome Mobile WebView,Android,c31b4e,380ef0,e56e80,9f1218f,1,-3578.0,2164.0,...,0.476787,0.176838,0.144666,0.0,0.109932,0.082522,0.189492,0.323432,0.000000,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499995,fffec77a1805ac2833dc4d1d0b13aade,Chrome,Windows,11f5678,NaN,a3f791,9ac1e4f,1,6616.0,-5747.0,...,0.480009,0.184745,0.207115,0.0,0.294988,0.180558,0.209586,0.560682,0.000000,0.0
499996,ffff1418e9af7bc4f31cb25f8c559f34,Chrome,Android,c31b4e,97023a,e56e80,9b08d64,1,11632.0,13288.0,...,0.386314,0.197803,0.000000,0.0,0.100788,0.000000,0.000000,0.343288,0.134796,1.0
499997,ffffc3f4d7fe28c39ec794571ac47694,Yandex Browser,Android,c31b4e,8ccc01,e56e80,72879b4,1,5611.0,10519.0,...,0.141758,0.372263,0.015989,0.0,0.000000,0.069675,0.119629,0.351251,0.000000,1.0
499998,ffffcc389b0db46598e56a268eb9d596,Yandex Browser,Windows,c31b4e,8ccc01,e56e80,6972223,2,12643.0,1469.0,...,0.534871,0.348122,0.130597,0.0,0.012910,0.000000,0.134230,0.156641,0.000000,0.0


In [38]:
test_features

,user_id,browser,os,country_id,region_id,timezone_id,domain,event_count,component0,component1,...,bb_22,bb_23,bb_24,bb_25,bb_26,bb_27,bb_28,bb_29,bb_30,bb_31
0,000098f8aa8b68a125148caff0a02827,Chrome,Windows,e48e27,NaN,f0c3df,b53adc6,1,15633.0,22503.0,...,0.395369,0.223871,0.366513,0.000000,0.000000,0.000000,0.000000,0.000000,0.377329,0.000000
1,0000eae7d06e8c6033731d1c1ba0c382,Chrome,Android,c31b4e,37246e,b4a275,72879b4,1,1489.0,15548.0,...,0.185387,0.198332,0.268159,0.000000,0.000000,0.000000,0.000000,0.124253,0.235184,0.000000
2,0000f7fa001a8d49b8e83b91a1215e17,Chrome,Windows,9bd92f,NaN,100cd5d,b6630fd,1,18530.0,14117.0,...,0.353522,0.099013,0.284027,0.118770,0.000000,0.000000,0.000000,0.047473,0.223296,0.000000
3,0001c39409954fe4b4148a23154fa905,Chrome Mobile,Android,c31b4e,380ef0,e56e80,635e50c,1,15290.0,-740.0,...,0.280179,0.576092,0.221572,0.324951,0.000000,0.143671,0.223182,0.236112,0.259828,0.000000
4,00029eeaef3671876a5d119acff7e792,Chrome Mobile,Android,c31b4e,f66ff,f6155e,7f91a1d,1,19051.0,9021.0,...,0.237485,0.373404,0.003393,0.248822,0.000000,0.175628,0.292717,0.140843,0.208554,0.085369
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84995,fffbf68a27011fbe76a7a856649c888f,Opera,Windows,11f5678,NaN,a3f791,b6630fd,1,19153.0,13108.0,...,0.291213,0.250462,0.180932,0.080350,0.010151,0.000000,0.000000,0.000000,0.147686,0.000000
84996,fffbf8879e9a31b944237fc24895aead,Yandex Browser,Windows,c31b4e,470e75,f6155e,8807153,1,11243.0,8618.0,...,0.625087,0.363995,0.690266,0.283356,0.000000,0.127537,0.095235,0.259695,0.373441,0.000000
84997,fffc4301db1d1760389be359a13bc740,Samsung Internet,Android,b98648,241b43,ad8d84,8d7355d,1,16429.0,13855.0,...,0.142557,0.280618,0.085487,0.000000,0.000000,0.000000,0.000000,0.000000,0.182319,0.000000
84998,fffd7f18a1234ea36ad7ceeeb50cf7ab,Yandex Browser,Windows,c31b4e,1fbfa5,e56e80,8807153,1,11243.0,8618.0,...,0.612064,0.469355,0.649910,0.274515,0.000000,0.136193,0.100409,0.283297,0.270754,0.000000


In [39]:
train_features.to_csv("preprocessed/train_with_bb_features.csv", index=False)
test_features.to_csv("preprocessed/test_with_bb_features.csv", index=False)

In [48]:
model.eval()

with torch.inference_mode():
    logits = model(df_test_cat_x_t.to(device), df_test_num_x_t.to(device)).cpu().numpy()

In [49]:
logits

array([ 2.0709019 ,  1.9223927 ,  2.4516501 , ...,  1.5697755 ,
       -6.3130965 ,  0.35489792], dtype=float32)

In [50]:
probs = 1 / (1 + np.exp(-logits))

In [52]:
pd.DataFrame(
    {"user_id;target": df_test_user_id.astype(str) + ";" + pd.Series(probs).astype(str)}
).to_csv("predictions/torch_prediction.csv", index=False)